## Install the dependencies



In [18]:
!pip install scikit-learn nltk


In [2]:
!pip install openai
!pip install -q git+https://github.com/huggingface/peft.git transformers bitsandbytes datasets


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 50.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.1/314.1 kB 32.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 14.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 14.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1

## Run the few_shot_instruction_generation.py and fine_tuned_blip_inference.py scripts

In [20]:
%run few_shot_instruction_generation.py
%run fine_tuned_blip_inference.py

## Pas the input image to the Grasp Point Genration model


In [30]:
#place holder code for the grasp point generation model
cup_base = Image.open("/content/drive/MyDrive/LLM final project/sample images/cup_base.jpeg")
cup_handle = Image.open("/content/drive/MyDrive/LLM final project/sample images/cup_handle2.jpeg")
cup_body = Image.open("/content/drive/MyDrive/LLM final project/sample images/cup_body.jpeg")

grasp_points = [cup_base, cup_handle, cup_body]



## Use the user input prompt to the get the grasping instructions and the grasping location


In [26]:
user_input = "i want tea"

instructions = generate_instruction(user_input) #get the instructions

print("Generated instructions based on the user input:\n",instructions)


Generated instructions based on the user input:
 task: pick up and bring the tea cup to the user  
object: tea cup  
grasping location: tea cup's handle


In [24]:
#sperate the grasping location

parts = instructions.split("Grasping location:")

# Extract the task details and grasping location
task_details = parts[0].strip()
grasping_location = "grasping location a " + parts[1].strip()
print(grasping_location)

grasping location a wine glass's stem


## Pas the grasp points to the captioning model and calculate the cosine similarity between the captions and the grasping location



In [31]:
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
scoring_model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

def get_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        outputs = scoring_model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.numpy()


In [32]:

grasp_scores = []
for image in grasp_points:
  caption = generate_caption(image)
  print(caption)
  # Get embeddings for the caption and the grasping location
  caption_embedding = get_embedding(caption)
  grasping_location_embedding = get_embedding(grasping_location)

  # Calculate cosine similarity
  cosine_sim = cosine_similarity(caption_embedding, grasping_location_embedding)
  grasp_scores.append([caption, cosine_sim[0][0]])
  print("Cosine Similarity:", cosine_sim[0][0])




grasping location a cup's base
Cosine Similarity: 0.5499435
grasping location a cup's handle
Cosine Similarity: 0.5652995
grasping location a cup's body
Cosine Similarity: 0.5003445


### Find the best grasp score

In [34]:
highest_score_item = max(grasp_scores, key=lambda x: x[1])

# Print the result
print("\nItem with the highest cosine similarity:")
print(highest_score_item[0])
print("Cosine Similarity:", highest_score_item[1])


Item with the highest cosine similarity:
grasping location a cup's handle
Cosine Similarity: 0.5652995
